In [ ]:
import pandas as pd
import time
import json
from google import genai
from pydantic import BaseModel
from typing import Optional

# 1. Initialize the 2026 Client
client = genai.Client(api_key="...")

# 2. Define the Scorecard Schema
class EvaluationResult(BaseModel):
    accuracy: int
    clarity: int
    completeness: int
    coherence: int
    relevance: int
    fluency: int
    hallucination: int
    reasoning: str

def judge_output(prompt_text, expected, generated):
    try:
        # In 2026, we use client.models.generate_content
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            contents=[
                f"PROMPT: {prompt_text}",
                f"EXPECTED: {expected}",
                f"GENERATED: {generated}"
            ],
            config={
                "system_instruction": "You are a strict LLM judge. Evaluate the generated output against the expected output.",
                "response_mime_type": "application/json",
                "response_schema": EvaluationResult,
            }
        )
        # The new SDK parses the JSON directly into the Pydantic model
        return response.parsed.model_dump() 
    except Exception as e:
        print(f"Error judging item: {e}")
        return None

# --- Main Execution ---

# Load your dataset
df = pd.read_json("./model_responses/prompt_outputs_alpaca.jsonl", lines=True, orient="records")
dataset = df.to_dict(orient="records")

results = []
requests_per_minute = 5 
delay_seconds = 60 / requests_per_minute

print(f"Starting evaluation of {len(dataset)} items...")

for i, data in enumerate(dataset):
    print(f"Judging item {i+1}/{len(dataset)}...")
    
    # Map your JSONL keys correctly
    verdict = judge_output(
        data.get('prompt'), 
        data.get('expected'), 
        data.get('model_response')
    )
    
    if verdict:
        results.append({**data, "evaluation": verdict})
    
    if i < len(dataset) - 1:
        time.sleep(delay_seconds)

# Save results
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Evaluation complete! Results saved to evaluation_results.json")

Starting evaluation of 19 items...
Judging item 1/19...
Judging item 2/19...
Judging item 3/19...
Judging item 4/19...
Judging item 5/19...
Judging item 6/19...
Judging item 7/19...
Judging item 8/19...
Judging item 9/19...
Judging item 10/19...
Judging item 11/19...
Judging item 12/19...
Judging item 13/19...
Judging item 14/19...
Judging item 15/19...
Judging item 16/19...
Judging item 17/19...
Judging item 18/19...
Judging item 19/19...
Evaluation complete! Results saved to evaluation_results.json
